In [9]:
import pandas as pd
import numpy as np

# ---------------- CONFIG ----------------
INPUT_PATH = r"C:\Users\USER\Desktop\Sales_Forecasting-Inventory_Management\data\Walmart_modified_with_anomalies.csv"
OUTPUT_PATH = r"C:\Users\USER\Desktop\Sales_Forecasting-Inventory_Management\data\Walmart_preprocessed_complete.csv"

# ---------------- LOAD DATA ----------------
df = pd.read_csv(INPUT_PATH)
print("Initial shape:", df.shape)

# ---------------- STANDARDIZE COLUMNS ----------------
df.columns = df.columns.str.strip().str.replace(" ", "_").str.replace("-", "_")

# ---------------- DATE COLUMN ----------------
date_cols = [c for c in df.columns if "date" in c.lower()]
if len(date_cols) == 0:
    raise ValueError("No date column found.")
DATE_COL = date_cols[0]
df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
df = df.dropna(subset=[DATE_COL])  # drop rows with invalid dates

# ---------------- NUMERIC AND CATEGORICAL ----------------
num_cols = df.select_dtypes(include=np.number).columns.tolist()
cat_cols = df.select_dtypes(include="object").columns.tolist()

# ---------------- HANDLE NEGATIVE VALUES ----------------
for col in num_cols:
    df.loc[df[col] < 0, col] = np.nan

# ---------------- FILL MISSING NUMERIC VALUES ----------------
for col in num_cols:
    if col in df.columns:
        df[col] = df.groupby(["Product_Name", "Brand", "Category"])[col].transform(lambda x: x.interpolate(method='linear', limit_direction='both'))
        df[col] = df[col].fillna(method='ffill').fillna(method='bfill')  # fill start/end NaNs

# ---------------- FILL MISSING CATEGORICAL VALUES ----------------
for col in cat_cols:
    df[col] = df[col].fillna("Unknown")

# ---------------- OUTLIER REMOVAL (CAPPING) ----------------
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df[col] = np.clip(df[col], lower, upper)  # cap extreme values

# ---------------- OPTIONAL SMOOTHING ----------------
TARGET_COL = "Market_Price" if "Market_Price" in df.columns else num_cols[0]
df[f"{TARGET_COL}_Smoothed"] = df[TARGET_COL].rolling(5, min_periods=1).mean()

# ---------------- SAVE ----------------
df.to_csv(OUTPUT_PATH, index=False)
print(f"Preprocessed dataset saved at: {OUTPUT_PATH}")
print("Final shape:", df.shape)
print("Any missing values remaining?", df.isna().sum().sum())


Initial shape: (52500, 20)


C:\Users\USER\AppData\Local\Temp\ipykernel_16136\580874036.py:35: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[col] = df[col].fillna(method='ffill').fillna(method='bfill')  # fill start/end NaNs
C:\Users\USER\AppData\Local\Temp\ipykernel_16136\580874036.py:35: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[col] = df[col].fillna(method='ffill').fillna(method='bfill')  # fill start/end NaNs
C:\Users\USER\AppData\Local\Temp\ipykernel_16136\580874036.py:35: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[col] = df[col].fillna(method='ffill').fillna(method='bfill')  # fill start/end NaNs
C:\Users\USER\AppData\Local\Temp\ipykernel_16136\580874036.py:35: FutureWarning: Series.fillna with 'method' is deprecated and will rais

Preprocessed dataset saved at: C:\Users\USER\Desktop\Sales_Forecasting-Inventory_Management\data\Walmart_preprocessed_complete.csv
Final shape: (51413, 21)
Any missing values remaining? 0
